# **ENTREGA FINAL -- Exp_51 (BO) + Produccion Agosto, en un unico notebook self-contained**

## 1. SETEOS INICIALES

### 1.1  Seteo Ambiente Local

In [ ]:
# Rutas locales (ya no hace falta Colab, Drive ni wget)
if (!exists("PARAM")) PARAM <- list()
PARAM$data_dir <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Clase1"

archivo_crudo    <- file.path(PARAM$data_dir, "competencia_01_crudo.csv")
archivo_ternaria <- file.path(PARAM$data_dir, "competencia_01.csv.gz")

if (!file.exists(archivo_crudo)) {
  stop("No encuentro el archivo crudo en: ", archivo_crudo)
}

### 1.2 Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type -> R

In [ ]:
require( "data.table" )

# leo el dataset
dataset <- fread(archivo_crudo)

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

# grabo local, en vez de /content/datasets/
fwrite( dataset,
    file = archivo_ternaria,
    sep = ","
)


In [ ]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

## 2 OPTIMIZACIÓN BAYESIANA

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

### 2.2 Carga de Librerias

Esta parte lleva  demasiados   minutos la primera vez en Google Colab
<br> por suerte a partir del 15-septiembre empezaremos a trabajar con Google Cloud, en donde todas las librerias, de Python R y Julia ya estarán instaladas

In [ ]:
# instalo paquetes
install.packages( c("R.utils", "rlist" ) )

In [ ]:
# Librerias  modernas  para la Bayesian Optimization
install.packages( c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging") )

In [ ]:
# el LightGBM
install.packages(c("lightgbm"))

In [ ]:
library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
# library(paradox)
library(DiceKriging)
library(lightgbm)

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- "Exp_51_entrega_final"
PARAM$semilla_primigenia <-  420029


In [ ]:
# training y future
PARAM$training_pct <- 70
PARAM$train <- c(202103, 202104)  # marzo + abril, como Exp_01
PARAM$train_final <- c(202103, 202104, 202105, 202106)  # marzo+abril+mayo+junio: todo lo conocido antes de agosto
PARAM$future <- c(202108)  # AGOSTO -- el mes real de la entrega
# PARAM$cortes: no aplica -- agosto no tiene clase_ternaria real, no hay curva de ganancia posible

In [ ]:
# Sin undersampling (tip del profe: pocos meses, no hace falta reducir los CONTINUA)
# entra el 100% del fold de training, tanto en la BO como en la produccion final

In [ ]:
PARAM$hyperparametertuning$iteraciones <- 10 # iteraciones bayesianas

In [ ]:
# Parametros LightGBM

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,     # fijo, esto es FUNDAMENTAL
  feature_fraction= 0.50,  # fijo
  learning_rate= 0.005,    # fijo

  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)


Aqui se definen los hiperparámetros de LightGBM que participan de la Bayesian Optimization

In [ ]:
# Aqui se cargan los bordes de los hiperparametros de la BO
#  es vital entender que se optimiza min_sum_hessian_in_leaf y que  min_data_in_leaf se deja fijo en CERO

PARAM$hypeparametertuning$hs <- ps(
  num_iterations= p_int(lower= 64L,  upper= 4096L),
  num_leaves= p_int(lower= 16L,  upper= 2048L),
  min_sum_hessian_in_leaf= p_dbl( lower=1e-06, upper=0.1)
)

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [ ]:
# logueo al archivo BO_log.txt
loguear  <- function( preg, arch=NA, verbose=TRUE )
{
  t0 <- Sys.time()
  reg <- copy( preg )
  if( "feature_contri" %in%  names(reg) ) {
    reg$feature_contri <- reg$feature_contri[1]
  }
  archivo <- arch
  if( is.na(arch) ) archivo <- paste0( folder, substitute( reg), ext )


  if( !file.exists( archivo ) )
  {
    # Escribo los titulos
    linea  <- paste0( "fecha\t",
                      paste( list.names(reg), collapse="\t" ), "\n" )

    cat( linea, file=archivo )
  }

  # escribo el registro
  linea  <- paste0( format(t0, "%Y%m%d.%H%M%S"),  "\t",     # la fecha y hora
                    gsub( ", ", "\t", toString( reg ) ),  "\n" )

  cat( linea, file=archivo, append=TRUE )  # grabo al archivo

  if( verbose )  cat( linea )   # imprimo por pantalla
}

### 2.2.4  Preprocesamiento

In [ ]:
# carpeta de trabajo
PARAM$exp_dir <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Competencia_1_Final"

setwd(PARAM$exp_dir)
dir.create(PARAM$experimento, showWarnings=FALSE)
setwd( file.path(PARAM$exp_dir, PARAM$experimento))

In [ ]:
# lectura del dataset
archivo_ternaria <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Clase1/competencia_01.csv.gz"
dataset <- fread(archivo_ternaria, stringsAsFactors= TRUE)

#### Correccion de datos + Lag1/Lag2/Delta1/Delta2 (variables originales, SIN Ranking Cero Fijo)

In [ ]:
# ===== Correccion de datos: ccajas_depositos colapsa a un unico valor (0) en 202105 =====
# (bug de carga, no efecto economico - visto en Exp_25 / Analisis_Estabilidad_Mensual)
dataset[foto_mes == 202105, ccajas_depositos := NA]

# ===== LAG 1, LAG 2 y sus DELTA -- sobre TODO el dataset =====
setorder(dataset, numero_de_cliente, foto_mes)

campos_lagueables <- setdiff(colnames(dataset),
                              c("numero_de_cliente", "foto_mes", "clase_ternaria",
                                "cprestamos_personales", "mprestamos_personales"))

dataset[, paste0(campos_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = campos_lagueables]
dataset[, paste0(campos_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = campos_lagueables]

for (campo in campos_lagueables) {
  dataset[, paste0(campo, "_delta1") := get(campo) - get(paste0(campo, "_lag1"))]
  dataset[, paste0(campo, "_delta2") := get(campo) - get(paste0(campo, "_lag2"))]
}

gc(full = TRUE, verbose = FALSE)

In [ ]:
dataset_train <- dataset[foto_mes %in% PARAM$train]

In [ ]:
  particionar(dataset_train,
    division= c(PARAM$training_pct, 100L -PARAM$training_pct),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_primigenia # aqui se usa SU semilla
  )

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forman parte del training
# SIN undersampling: entra el 100% del fold de training

dataset_train[, training := 0L]
dataset_train[
  foto_mes %in%  PARAM$train &  fold==1,
  training := 1L
]

In [ ]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "fold", "training")
)

#### 2.2.4.1  Salsa Magica (dedicada a Laura Pasquini)

Los EDAs que estan haciendo los alumnos de los lunes son pura falopa, y de la mala. El uso de Agentes de la IA solo les empera el daño cerebral, generan pedf larguisimos, que no te dan ganas de leerlos, carentes de insights y alma.
<br>El unico EDA que tiene realmente sentido, en este caso, es comparar la distribución de cada variable de los meses donde se entrena (202103 a 202106) y el mes donde se va a aplicar el modelo (202108), buscando Data Drifting.
<br>Aun con los reales datos del futuro, donde no se tiene la clase, SIEMPRE se puede analizar el Data Drifting.
<br>Para las variables con distribuciones distintas se puede hacer:
* Transformar la variable para hacerla homogenea entre los dos periodos
* Eliminarla (es lo simple, que se eligió hacer en este caso)

En esa exploracion gráfica del Data Drifting, se vio que estas variables cambian de distribucion entre los meses de entrenamiento y el mes futuro, y se decidió brutalmente eliminarlas
* cprestamos_personales
* mprestamos_personales


más adelantes, se utilizarán técnicas más inclusivas, y se preservaran dichas variables, transformadas.


In [ ]:
campos_buenos <- setdiff( campos_buenos, c("cprestamos_personales", "mprestamos_personales") )

2.2.5 Configuracion Bayesian Optimization

In [ ]:
giter <- 0

if( file.exists("BO_log.txt") ){
  tb_BO <- fread("BO_log.txt")
  giter <- tb_BO[, max(iter)]
}


In [ ]:
# En el argumento x llegan los parametros de la bayesiana

Estimar_lightgbm <- function(xs) {

  flush.console()
  giter <<- giter + 1

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, xs)
  xx <- copy(param_completo)

  # entreno LightGBM
  modelo <- lgb.train(
    data= dtrain,
    valids= list( "val"=dvalidate),
    param= param_completo
  )


  # obtengo la ganancia
  xx$iter <- giter
  xx$metrica <- modelo$best_score

  loguear( xx, "BO_log.txt")
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")  # le reordeno a mlr3MBO

  return( list( metrica = xx$metrica) )
}

In [ ]:
# dejo los datos de training en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

In [ ]:
# datos de validation en el formato que necesita LightGBM
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 0L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 0L, clase01],
  free_raw_data= FALSE
)

nrow(dvalidate)
ncol(dvalidate)

In [ ]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- Estimar_lightgbm # la funcion que voy a maximizar

objective <- ObjectiveRFun$new(
  fun= funcion_optimizar,
  domain= PARAM$hypeparametertuning$hs,
  codomain= ps(metrica = p_dbl(tags="maximize")),
  id= "BO_lightgbm"
)


instance <- OptimInstanceBatchSingleCrit$new(
  objective= objective,
  terminator=  trm("evals", n_evals = PARAM$hyperparametertuning$iteraciones)
)


surrogate <- srlrn(lrn("regr.km",
  optim.method = "BFGS",
  nugget.estim = TRUE,
  control= list(trace = FALSE)))

acq_function  <- acqf("ei")                 # expected improvement

acq_optimizer  <- acqo(
  optimizer  = opt("random_search", batch_size= 1000),
  terminator = trm("evals", n_evals= 1000)
)

optimizer <- opt("mbo",
  loop_function = bayesopt_ego,
  surrogate= surrogate,
  acq_function= acq_function,
  acq_optimizer= acq_optimizer
)



2.2.6 Corrida Bayesian Optimization

In [ ]:
bayesiana_salida <- optimizer$optimize(instance)

Luego de este punto, en la carpeta del experimento quedó el archivo BO_log.txt que posee todas las iteraciones de la Bayesian Optimization

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
setwd(PARAM$exp_dir)
dir.create(PARAM$experimento, showWarnings= FALSE)
setwd( file.path(PARAM$exp_dir, PARAM$experimento ))

In [ ]:
 # cergo los mejores hiperparametros, que son los de mayor metrica
 tb_bayesiana <- fread("BO_log.txt")
 setorder( tb_bayesiana, -metrica )
 PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_bayesiana[1])

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la  optimización bayesiana

In [ ]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N, list(foto_mes, clase_ternaria)]

#### Final Training Hyperparameters

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

In [ ]:
# ===== SEMILLERIO: 20 semillas (5 habituales + 15 primos nuevos) =====
PARAM$semillerio <- c(420029, 425207, 420521, 420307, 420929,
                       500009, 523021, 546031, 569047, 592049,
                       615053, 638059, 661061, 684091, 707099,
                       730111, 753127, 776137, 799147, 822161)

dfuture <- dataset[foto_mes %in% PARAM$future]
predicciones <- matrix(nrow = nrow(dfuture), ncol = length(PARAM$semillerio))

In [ ]:
# Entreno los 20 modelos (mismos hiperparametros, solo cambia la semilla)

hora_inicio_semillerio <- Sys.time()

for (i in seq_along(PARAM$semillerio)) {

  t0_semilla <- Sys.time()
  param_final$seed <- PARAM$semillerio[i]

  dtrain_final <- lgb.Dataset(
    data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
    label= dataset_train[, clase01]
  )

  modelo_final <- lgb.train(
    data= dtrain_final,
    param= param_final
  )

  predicciones[, i] <- predict(modelo_final, data.matrix(dfuture[, campos_buenos, with= FALSE]))

  duracion_semilla <- round(difftime(Sys.time(), t0_semilla, units = "secs"), 1)
  transcurrido <- round(difftime(Sys.time(), hora_inicio_semillerio, units = "mins"), 1)
  cat(sprintf("[%s] semilla %d/%d (%d) lista - tardo %ss - transcurrido %smin\n",
              format(Sys.time(), "%H:%M:%S"), i, length(PARAM$semillerio),
              PARAM$semillerio[i], duracion_semilla, transcurrido))
}

### 3.1 Exportacion de probabilidades: una por semilla + tabla unificada (22 columnas)

In [ ]:
dir.create("prob_individuales", showWarnings = FALSE)

# 1) un archivo por semilla (id, mes, prob)
for (i in seq_along(PARAM$semillerio)) {
  tb_i <- data.table(
    numero_de_cliente = dfuture$numero_de_cliente,
    foto_mes = dfuture$foto_mes,
    prob = predicciones[, i]
  )
  fwrite(tb_i,
    file = file.path("prob_individuales", sprintf("prediccion_semilla_%d.txt", PARAM$semillerio[i])),
    sep = "\t"
  )
}

# 2) unificacion: ID + MES + las 20 probabilidades = 22 columnas
tb_probabilidades <- data.table(
  numero_de_cliente = dfuture$numero_de_cliente,
  foto_mes = dfuture$foto_mes
)
for (i in seq_along(PARAM$semillerio)) {
  tb_probabilidades[, paste0("prob_semilla_", PARAM$semillerio[i]) := predicciones[, i]]
}

cat("Columnas de la tabla unificada:", ncol(tb_probabilidades), "\n")  # 2 + 20 = 22

fwrite(tb_probabilidades, file = "probabilidades_20_semillas.txt", sep = "\t")

# el ensamble (promedio simple) se calcula aparte, no se agrega como columna 23
prob_ensamble <- rowMeans(predicciones)

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")